# SAFOD DAS + repeating earthquakes: advisor checkpoint

**Current decision: PROCEED_WITH_CONDITIONS toward an incremental-value test.** The published-label repair is complete, and the deep fiber clearly detects a prospective 2026 target-family candidate. No DAS catalog-extension, stress-drop, or creep-rate claim has passed.

The original phrase “25 verified members” was wrong. Those rows are single-anchor high-similarity candidates. Two exact-ID matches belong to other published families, proving that the one-seed procedure overmerged. All DAS spatial axes remain OptaSense locus index, not depth, until surveyed geometry is supplied.

In [ ]:
# Advisor controls: edit this cell, then Run All.
from pathlib import Path
import sys

candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    if (base / 'config' / 'pilot.json').exists() and base.name == 'repeaters_v2':
        candidates.append(base)
    nested = base / 'faultzone' / 'repeaters_v2'
    if (nested / 'config' / 'pilot.json').exists():
        candidates.append(nested)
PROJECT = next((path.resolve() for path in candidates), None)
if PROJECT is None:
    raise FileNotFoundError('Could not locate faultzone/repeaters_v2')
NOTEBOOK_ROOT = PROJECT.parents[1]
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
REBUILD_RAW = False
print('Project:', PROJECT)
print('Cached mode:', not REBUILD_RAW)

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from faultzone.repeaters_v2.src.checkpoint import write_checkpoint

checkpoint = write_checkpoint(PROJECT)
branches = pd.DataFrame(checkpoint['branches'])
milestones = pd.DataFrame(checkpoint['checkpoints'])
display(Markdown('## Decision: {}'.format(checkpoint['pilot_decision'])))
display(Markdown(checkpoint['decision_basis']))
symbols = {'PASS': 'PASS', 'CONDITIONAL': 'CONDITIONAL', 'STOP': 'STOP'}
view = branches[['branch', 'status', 'evidence', 'next_gate']].copy()
view.insert(0, 'gate', view['status'].map(symbols))
display(view)
display(Markdown('### Checkpoints'))
display(milestones)

## Label repair and external validation

Historical labels come from exact event IDs in Waldhauser--Schaff (2021), not from NCEDC proximity or a single correlation threshold. The frozen published population contains six target-sequence events and 12 events in three neighboring sequences.

In [ ]:
external = json.load(open(PROJECT / 'outputs' / 'external_validation' / 'status.json'))
crosswalk = pd.read_csv(PROJECT / 'outputs' / 'external_validation' / 'shortlist_external_crosswalk.csv')
diagnostic = pd.read_csv(PROJECT / 'outputs' / 'external_validation' / 'single_anchor_repair_diagnostic.csv')
display(pd.DataFrame([external]).drop(columns=['catalog_provenance', 'hard_negative_sequence_ids'], errors='ignore'))
display(diagnostic)
matched = crosswalk[crosswalk['external_match_type'] == 'exact_event_id'].copy()
display(matched[['event_id', 'single_anchor_decision', 'single_anchor_median_correlation', 'external_sequence_id', 'external_role', 'diagnostic_outcome']])

## The 38-event table is a similarity shortlist, not a recurrence catalog

Orange rows abstain for insufficient HRSN data; gray rows have no downloadable window. Neither is a negative label. The timeline cannot support creep rate because nomination and recurrence completeness are unqualified.

In [ ]:
shortlist = pd.read_csv(PROJECT / 'outputs' / 'hrsn' / 'similarity_shortlist_v2.csv')
shortlist['time'] = pd.to_datetime(shortlist['origin_time'], utc=True)
display(shortlist[['event_id', 'origin_time', 'magnitude', 'distance_3d_m', 'decision', 'membership_status', 'catalog_status', 'overall_median_correlation']])
colors = {'unlabeled_deep_seed': 'tab:blue', 'single_anchor_high_similarity': 'tab:green', 'insufficient_data': 'tab:orange', 'single_anchor_dissimilar_neighbor': 'tab:red'}
plot_cc = shortlist['overall_median_correlation'].fillna(1.0)
fig, ax = plt.subplots(figsize=(11, 3.8), constrained_layout=True)
for status, group in shortlist.assign(plot_cc=plot_cc).groupby('membership_status'):
    ax.scatter(group['time'], group['plot_cc'], s=40 + 18 * group['magnitude'], color=colors.get(status, '0.5'), label=status)
ax.axhline(0.85, color='tab:red', linestyle='--', linewidth=1)
ax.set_ylim(0.8, 1.01)
ax.set_ylabel('Single-anchor median HRSN correlation')
ax.set_title('Frozen 38-event similarity shortlist — not a family catalog')
ax.legend(frameon=False, ncol=2)
ax.grid(alpha=0.2)
plt.show()

## What is measured, and what is next

The 2026 candidate is detected over the sampled deep-fiber aperture and the same-fiber control supplies a frozen negative distribution. The HRSN multi-family correlation diagnostic classified zero of ten available labeled events under its frozen margin. This motivates testing DAS spatial features, but it is not evidence that DAS wins. The next fair step is network-only continuous detection and differential relocation on the exact DAS intervals, followed by blinded DAS-only and joint pipelines.

In [ ]:
deep = pd.read_csv(PROJECT / 'outputs' / 'deep_das' / 'event_metrics.csv')
network = json.load(open(PROJECT / 'outputs' / 'external_validation' / 'network_family_status.json'))
claims = pd.read_csv(PROJECT / 'outputs' / 'external_validation' / 'claim_status.csv')
display(deep[['event_id', 'role', 'status', 'median_channel_snr', 'detected_block_fraction', 'usable_power_snr_ranges_hz']])
display(pd.DataFrame([network]).drop(columns=['per_family'], errors='ignore'))
display(claims)
print('Highest-value next analysis:', checkpoint['highest_value_next_analysis'])
print('Highest-value next observation:', checkpoint['highest_value_next_observation'])
print('Highest-value metadata request:', checkpoint['highest_value_metadata_request'])

## Raw rebuild is explicit

Leave REBUILD_RAW false for an advisor session. A true value reruns official/read-only inputs and may require network access and several minutes. It still does not run the future continuous DAS/network scans.

In [ ]:
if REBUILD_RAW:
    import subprocess
    commands = [
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.build_coverage_ledger'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.run_hrsn_pilot', '--max-family-candidates', '38'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.run_deep_das_pilot'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.run_active_source_calibration'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.build_external_validation'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.run_network_family_benchmark'],
        [sys.executable, '-m', 'faultzone.repeaters_v2.src.checkpoint'],
    ]
    for command in commands:
        print('RUN', ' '.join(command))
        subprocess.run(command, cwd=NOTEBOOK_ROOT, check=True)
else:
    print('Cached-only checkpoint complete; no raw files or network were touched.')